# Experiment 8: Semantic Similarity

- **WordNet:** words grouped into meanings (synsets) in a tree (dog -> canine -> animal).
  - Path similarity = 1 / (1 + shortest path)
  - Wu-Palmer = 2 * depth(common ancestor) / (depth(a) + depth(b))
- **GloVe word vectors:** similar contexts -> similar vectors. Compare with cosine similarity.
- Vectors rate antonyms (hot/cold) as similar, because they appear in the same contexts.

In [ ]:
# !pip install nltk gensim scipy pandas

In [ ]:
import nltk
import numpy as np
import pandas as pd
import gensim.downloader as api
from scipy.stats import spearmanr
from nltk.corpus import wordnet as wn, stopwords
from gensim.test.utils import datapath

for r in ['wordnet', 'omw-1.4', 'stopwords']:
    nltk.download(r, quiet=True)

glove = api.load('glove-wiki-gigaword-100')   # ~128 MB download, first time only
stop = set(stopwords.words('english'))

## Exploring WordNet

In [ ]:
for s in wn.synsets('bank')[:4]:          # one word, many meanings
    print(s.name(), '-', s.definition())

dog = wn.synset('dog.n.01')
print('\nSynonyms :', dog.lemma_names())
print('Hypernyms:', ' -> '.join(s.name().split('.')[0] for s in dog.hypernym_paths()[0]))

## WordNet vs GloVe on word pairs

In [ ]:
def wordnet_sim(w1, w2):
    # best Wu-Palmer score over all noun meanings
    return max((a.wup_similarity(b) or 0) for a in wn.synsets(w1, 'n') for b in wn.synsets(w2, 'n'))

pairs = [('car', 'automobile'), ('dog', 'cat'), ('coffee', 'tea'), ('coffee', 'cup'),
         ('doctor', 'nurse'), ('computer', 'banana')]

pd.DataFrame([{'Word 1': a, 'Word 2': b,
               'WordNet (Wu-Palmer)': round(wordnet_sim(a, b), 3),
               'GloVe (cosine)': round(float(glove.similarity(a, b)), 3)} for a, b in pairs])

## Analogies: king - man + woman = ?

In [ ]:
for a, b, c in [('man', 'king', 'woman'), ('france', 'paris', 'india'), ('walk', 'walked', 'swim')]:
    answer = glove.most_similar(positive=[b, c], negative=[a], topn=1)[0][0]
    print(f'{a} : {b} :: {c} : {answer}')

In [ ]:
# Antonyms get HIGH vector similarity
for a, b in [('hot', 'cold'), ('good', 'bad')]:
    print(a, b, round(float(glove.similarity(a, b)), 3))

## Sentence similarity (average of word vectors)

In [ ]:
def sent_vec(text):
    words = [w for w in text.lower().split() if w.isalpha() and w not in stop and w in glove]
    return np.mean([glove[w] for w in words], axis=0)

def cosine(u, v):
    return float(u @ v / (np.linalg.norm(u) * np.linalg.norm(v)))

for s1, s2 in [('A man is playing a guitar', 'A person is playing a musical instrument'),
               ('The stock market fell today', 'The chef cooked a pasta dinner'),
               ('The dog bit the man', 'The man bit the dog')]:   # word order is ignored!
    print(f'{cosine(sent_vec(s1), sent_vec(s2)):.3f}  {s1} | {s2}')

## Evaluation against human scores (WordSim-353)

In [ ]:
ws = pd.read_csv(datapath('wordsim353.tsv'), sep='\t', comment='#', header=None, names=['w1', 'w2', 'human'])
ws = ws[[a.lower() in glove and b.lower() in glove for a, b in zip(ws.w1, ws.w2)]]

glove_scores = [glove.similarity(a.lower(), b.lower()) for a, b in zip(ws.w1, ws.w2)]
print("GloVe Spearman's rho:", round(spearmanr(ws.human, glove_scores)[0], 3))